In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from scripts.denoising import *
from scripts.TPS import *

In [ ]:
X = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_position_matrix.csv"))
Y = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_noisy_velocity_matrix.csv"))
t = pd.read_csv("./data/s_curve/uniform/s_curve_gt_latent_time_vector.csv")
t = list(t["t"])

X_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_position_matrix.csv"))
Y_gt = np.array(pd.read_csv("./data/s_curve/uniform/s_curve_gt_velocity_matrix.csv"))

# X = X_gt
# Y = Y_gt

X.shape, Y.shape

In [ ]:
# Visualize using the provided function
plot_3d_with_quiver(
    X,
    Y,
    t,
    arrow_size=0.2
)

In [ ]:
import umap

umap_reducer = umap.UMAP(n_neighbors=15, min_dist=0.5, n_components=2, random_state=42)
X_2d = umap_reducer.fit_transform(X)
plot_2d(X_2d, t, "UMAP")

In [ ]:
tps = ThinPlateSpline(X_2d, n_control_points=1000)
tps.fit(X, dof_target=50)

In [ ]:
metrics = tps.evaluate_fit(X)
metrics

In [ ]:
X_smoothed = tps.predict(X_2d)
plot_3d(X,t)

plot_3d(X_smoothed,t)

In [ ]:
import numpy as np

# Define grid resolution
num_grid_points = 30  # Adjust based on desired granularity

# Compute the min and max of X_2d
x_min, y_min = X_2d.min(axis=0)
x_max, y_max = X_2d.max(axis=0)

# Generate grid points
x_lin = np.linspace(x_min, x_max, num_grid_points)
y_lin = np.linspace(y_min, y_max, num_grid_points)
grid_x, grid_y = np.meshgrid(x_lin, y_lin)  # Create a 2D grid
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])  # Flatten grid

# Confirm it's not affecting TPS fitting
plot_2d(X_2d, t, "UMAP with Grid Points")
plot_2d(grid_points, np.zeros(grid_points.shape[0]), "Grid Points")

In [ ]:
tps = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
tps.control_points = grid_points
pairwise_distances = cdist(X_2d, tps.control_points, metric="euclidean")
tps.K = tps.tps_kernel(pairwise_distances)
tps.fit(X, dof_target=20)

In [ ]:
X_smoothed = tps.predict(X_2d)
plot_3d(X_smoothed,t)

In [ ]:
import plotly.graph_objects as go
import numpy as np

grid_3d = tps.predict(grid_points)
# Stack grid points and data points together
combined_3d = np.vstack([grid_3d, X_smoothed])
combined_colors = [0] * len(grid_3d) + list(t)  # Grey for grid, t for data

# Create an interactive scatter plot
fig = go.Figure()

# Add grid points (grey)
fig.add_trace(go.Scatter3d(
    x=grid_3d[:, 0], y=grid_3d[:, 1], z=grid_3d[:, 2],
    mode='markers',
    marker=dict(size=3, color='grey'),
    name='Grid Points'
))

# Add data points (colored by t)
fig.add_trace(go.Scatter3d(
    x=X_smoothed[:, 0], y=X_smoothed[:, 1], z=X_smoothed[:, 2],
    mode='markers',
    marker=dict(size=3, color=t, colorscale='Viridis', opacity=0.8),
    name='Data Points'
))

# Layout settings
fig.update_layout(
    title="Interactive 3D Plot of Grid and Data Points",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    )
)

# Show interactive plot
fig.show()


In [ ]:
jacobians = tps.compute_tps_jacobians(X_2d)
jacobians.shape

In [ ]:
tps_vf = ThinPlateSpline(X_2d, n_control_points=num_grid_points**2)
tps_vf.control_points = grid_points
pairwise_distances = cdist(X_2d, tps.control_points, metric="euclidean")
tps_vf.K = tps.tps_kernel(pairwise_distances)
tps_vf.fit(Y, dof_target=20)

In [ ]:
Y_smoothed = tps_vf.predict(X_2d)

projected_velocities = np.zeros_like(X_2d)
num_points = X_2d.shape[0]
for i in range(num_points):
    projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y_smoothed[i])
    
plot_2d_quiver(X_2d, projected_velocities, t, scale=5, cmap='coolwarm', arrow_color='black')

In [ ]:
projected_velocities = np.zeros_like(X_2d)
num_points = X_2d.shape[0]
for i in range(num_points):
    projected_velocities[i], _, _, _ = np.linalg.lstsq(jacobians[i], Y[i])

In [ ]:
plot_2d_quiver(X_2d, projected_velocities, t, scale=5, cmap='coolwarm', arrow_color='black')

In [ ]:
tps_vf = ThinPlateSpline(X_2d, n_control_points=1000)
tps_vf.fit(projected_velocities, dof_target=50)

In [ ]:
vf_2d_smoothed = tps_vf.predict(X_2d)

In [ ]:
plot_2d_quiver(X_2d, vf_2d_smoothed, t, scale=5, cmap='coolwarm', arrow_color='black')

In [ ]:
from scripts.steam_plot import *


plot_velocity_streamplot(X_2d, tps_vf, t, 20)

In [ ]:
import matplotlib.pyplot as plt

def plot_velocity_streamplot(X_2d, tps_vf, quiver_directions=None, 
                             scatter_color="blue", scatter_size=10, title=None):
    """
    Computes a velocity field on a grid, fills in the grid with predicted velocities,
    and plots a streamplot overlaying the original mesh grid points with customizable scatter color and size.
    Also overlays a quiver plot with user-provided quiver directions.

    Parameters
    ----------
    X_2d : np.ndarray
        A 2D numpy array of shape (n_points, 2) containing the original mesh grid points.
    tps_vf : object
        An object with a .predict() method that computes velocity given a point.
        Its predict() method should accept an array of shape (1, 2) and return an array of shape (1, 2).
    quiver_directions : np.ndarray or None, optional
        A 2D numpy array of shape (n_points, 2) specifying the quiver arrow directions for each X_2d point.
        If None, no quiver plot is added.
    scatter_color : str, optional
        Color of the scatter plot points (default is "blue").
    scatter_size : int or float, optional
        Size of the scatter plot points (default is 10).

    Returns
    -------
    None
    """
    # Compute the velocity on the grid
    X_grid = compute_velocity_on_grid(X_2d)

    # Extract unique x and y values from X_grid
    x_vals = np.unique(X_grid[:, 0])
    y_vals = np.unique(X_grid[:, 1])

    # Create a full mesh grid
    xx, yy = np.meshgrid(x_vals, y_vals)  # Shape: (ny, nx)

    # Create an empty velocity grid with NaN values
    ny, nx = xx.shape
    V_mesh = np.full((ny, nx, 2), np.nan)  # Holds the velocity field

    # Compute predicted velocity and store in V_mesh
    for point in X_grid:
        x_pt, y_pt = point[0], point[1]
        j_idx = np.where(x_vals == x_pt)[0]
        i_idx = np.where(y_vals == y_pt)[0]
        if j_idx.size > 0 and i_idx.size > 0:
            vel = tps_vf.predict(point.reshape(1, -1))[0]  # shape: (2,)
            V_mesh[i_idx[0], j_idx[0], :] = vel

    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Scatter plot of original points
    ax.scatter(X_2d[:, 0], X_2d[:, 1], s=scatter_size, c=scatter_color, alpha=0.5)

    # Streamplot
    ax.streamplot(x_vals, y_vals, V_mesh[:, :, 0], V_mesh[:, :, 1], color='k', density=1, arrowsize=1.5)

    # Add quiver plot if quiver_directions is provided
    if quiver_directions is not None:
        ax.quiver(X_2d[:, 0], X_2d[:, 1], quiver_directions[:, 0], quiver_directions[:, 1], 
                  color="red", scale=15, width=0.003, headwidth=3, alpha=0.8)
    if title:
        ax.set_title(title)
    else:
        ax.set_title("Streamplot with Quiver Overlay")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.show()


In [ ]:
plot_velocity_streamplot(X_2d, tps_vf, quiver_directions=jacobians[:,0,:], 
                         scatter_color=t, scatter_size=10)

In [ ]:
plot_velocity_streamplot(X_2d, tps_vf, quiver_directions=jacobians[:,1,:], 
                         scatter_color=t, scatter_size=10)

In [ ]:
plot_velocity_streamplot(X_2d, tps_vf, quiver_directions=jacobians[:,2,:], 
                         scatter_color=t, scatter_size=10)

In [ ]:
# Assuming jacobians has shape (N, d, D) and tps_vf.predict(X_2d) has shape (N, D)
vector_field = tps_vf.predict(X_2d)  # Shape: (N, D)
alignment_scores = []
N = X_2d.shape[0]

for i in range(jacobians.shape[1]):  # Iterate over Jacobian components
    jacobian_component = jacobians[:, i, :]  # Shape: (N, D)
    
    # Compute the dot product per row and sum
    numerator = np.sum(jacobian_component * vector_field, axis=1)
    
    # Compute normalization factor (L2 norms of each row, then sum over all rows)
    norm_factor = np.linalg.norm(jacobian_component, axis=1) * np.linalg.norm(vector_field, axis=1)
    
    # Avoid division by zero
    alignment_score = np.sum(np.abs(numerator / norm_factor)) / N
    alignment_scores.append(alignment_score)
    
    print(f"Normalized alignment score for Jacobian component {i}: {alignment_score}")

In [ ]:
import rpy2.robjects as ro
from rpy2.robjects import numpy2ri
from rpy2.robjects.packages import importr

# Enable conversion between NumPy and R objects
numpy2ri.activate()

# Import princurve with conflict resolution warning
princurve = importr("princurve", on_conflict="warn")

# Convert NumPy array to R-compatible format
X_2d_r = ro.r.matrix(X_2d, nrow=X_2d.shape[0], ncol=X_2d.shape[1])

# Fit principal curve
fit = princurve.principal_curve(X_2d)

# Print the fitted curve structure
print(fit)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

def plot_2d(points, points_color, principal_curve=None, title=""):
    """
    Plot 2D scatter points with an optional principal curve.

    Args:
        points (np.ndarray): 2D points as an array of shape (n, 2).
        points_color (list or np.ndarray): Colors corresponding to each point.
        principal_curve (np.ndarray or None): Principal curve points of shape (m, 2).
        title (str): Title of the plot.
    """
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="white", constrained_layout=True)
    fig.suptitle(title, size=12)  # Adjust title size for visibility
    ax.tick_params(axis='both', which='major', labelsize=8)  # Adjust tick label size

    # Scatter plot
    ax.scatter(points[:, 0], points[:, 1], c=points_color, alpha=0.6, label="Data Points")

    # Plot the principal curve if provided
    if principal_curve is not None:
        ax.scatter(principal_curve[:, 0], principal_curve[:, 1], color="red", lw=2, label="Principal Curve")

    # Set axis ticks
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    # Add legend if principal curve is plotted
    if principal_curve is not None:
        ax.legend()

    plt.show()


In [ ]:
# Assume X_2d contains 2D data points and princurve has been fitted
principal_curve_points = np.array(fit.rx2("s"))  # Extract principal curve from R

# Plot with principal curve
plot_2d(X_2d, points_color="blue", principal_curve=principal_curve_points, title="Principal Curve Fit")